In [21]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import udf, col
from pyspark.sql.types import FloatType
import numpy as np
from models.fcm import Dfcm
from utils.validity import * 
import time 

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("FCM_PySpark").getOrCreate()
spark

24/08/24 06:50:07 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 172.20.10.2 instead (on interface wlp6s0)
24/08/24 06:50:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/24 06:50:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Bước 1: Đọc và chuẩn bị dữ liệu
csv_file_path = "data/csv/602_Dry_Bean.csv"
data = spark.read.csv(csv_file_path, header=True, inferSchema=True)
data = data.drop(data.columns[-1])
for column in data.columns:
    data = data.withColumn(column, col(column).cast(FloatType()))


# Bước 1: Tính tổng các giá trị thuộc tính của một đối tượng và thêm giá trị tổng này thành cột mới vào tập dữ liệu. Thực hiện trên toàn bộ dữ liệu 
data = data.withColumn("sum", sum(col(column) for column in data.columns)) 

# Bước 2: Sắp xếp tập dữ liệu theo cột tổng mới tạo theo thứ tự tăng dần 
data = data.orderBy("sum", ascending=True) 

In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col, sum as _sum, row_number

# Bước 3: Chia tập dữ liệu theo chiều ngang thành k phần bằng nhau, ở đây chia làm 3 dataframe 
window = Window.orderBy("sum")
data_indexed = data.withColumn("row_index", row_number().over(window))

# Define the number of clusters (segments)
k = 7  

# Tính toán số lượng dòng của tập dữ liệu để phân đoạn 
total_rows = data_indexed.count()
segment_size = total_rows // k

# Bước 3: Chia tập dữ liệu theo chiều ngang thành k phần bằng nhau
segments = []
for i in range(k):
    start_idx = i * segment_size + 1
    end_idx = (i + 1) * segment_size
    if i == k - 1:  # Chắc chắn rằng tất cả các dòng đều được chia hết cho k
        end_idx = total_rows

    segment = data_indexed.filter((col("row_index") >= start_idx) & (col("row_index") <= end_idx)).drop("row_index")
    segments.append(segment)

In [ ]:
# Bước 4: Đối với mỗi phân đoạn, tính tổng các cột thuộc tính (không bao gồm các cột đã tạo ở bước 1)
# tính giá trị trung bình của nó và đặt vào trong hàng mới. Hàng mới này thực sự là 
# một trong những trọng tâm cụm đã khởi tạo

# Khởi tạo một list rỗng để lưu trữ các trọng tâm cụm   
cluster_centers = []

# Duyệt qua từng phân đoạn
for segment in segments:
    # Tính tổng các cột thuộc tính 
    segment_sums = segment.agg(*[_sum(col_name).alias(col_name) for col_name in data.columns[:-1]])
    
    # Tính giá trị trung bình của các cột thuộc tính
    row_count = segment.count()
    cluster_center = [segment_sums.select(col_name).first()[0] / row_count for col_name in data.columns[:-1]]
    
    # Thêm giá trị trung bình của các cột thuộc tính vào hàng mới
    cluster_centers.append(cluster_center)

print("Initialized Cluster Centers using Naive Sharding:")
for idx, center in enumerate(cluster_centers):
    print(f"Cluster Center {idx + 1}: {center}", len(center))

In [6]:
assembler = VectorAssembler(inputCols=data.columns[:-1], outputCol="features")
data = assembler.transform(data)

data_rdd = data.select("features").rdd.map(lambda row: row[0].toArray())

In [7]:
# Sau khi khởi tạo trọng tâm cụm với thuật toán sharding, sử dụng tâm cụm này là một biến toàn cục và được phát tới tất cả
# các nút công nhân (workers) thông qua broadcast trong Spark.
cluster_centers = spark.sparkContext.broadcast(cluster_centers)

In [8]:
# np.array(cluster_centers.value).shape   

In [9]:
# chia dữ liệu theo chiều ngang để phân bổ cho các nút công nhân
partitioned_data_rdd = data_rdd.repartition(k)
print("Số lượng phân vùng: ", partitioned_data_rdd.getNumPartitions())  

# # Optional: If you want to see the partition distribution, collect some samples
# samples = partitioned_data_rdd.mapPartitions(lambda it: [len(list(it))]).collect()
# print(f"Data distribution across partitions: {samples}")

Số lượng phân vùng:  7


---

## Khởi tạo tâm cụm

In [39]:
# Lấy dữ liệu trong từng phân vùng
partitioned_data = partitioned_data_rdd.glom().collect()

fcm = Dfcm()

_start_time = time.time()

v = np.array(cluster_centers.value)
for step in range(10000):
    v_old = v.copy() 
    Us = []
    for i, partition in enumerate(partitioned_data):
        data_np = np.array(partition)
        # print(data_np.shape)

        sdistances = norm_distances(data_np, v)
        membership = fcm.update_membership_matrix(sdistances)
        Us.append(membership)
    U_final = np.concatenate(Us)

    v = fcm.update_cluster_centers(np.array(data_rdd.collect()), U_final)
    print((np.abs(v - v_old)).max(axis=(0, 1)))
    if (np.abs(v - v_old)).max(axis=(0, 1)) < 1e-5:
        break
    
metric_nt = {
    'Time': round_float(time.time() - _start_time),
    'PC': partition_coefficient(U_final) ,
}
print(metric_nt)    